# Intelligent Vacuum-Cleaning Robot

## Group Members

| Seat No | Names |
| ------- | ----- |
| CS-24115 | Ukasha |
| CS-24117 | Abdul Mueed |
| CS-24123 | Abdullah Baig |
| CS-24127 | Umer Iqbal |

Plan: read the two text files -> CSP chooses the cleaning order -> BFS/A* drives the robot -> draw the results.

How to run: **Kernel > Restart & Run All** (from this project folder).

## 1. Read the input files

The files are read at runtime, so any valid `Environment.txt` / `Schedule.txt` works.
Coordinates are `(row, column)`, zero-based. The first dirty cell is `D1`, the second `D2`, etc.
Time slots are stored as plain numbers: `T3` becomes `3`.

In [23]:
import os
import re
from collections import deque

NUMBER_PAIR = r'\(?\s*(\d+)\s*,\s*(\d+)\s*\)?'   # (row, col)
WORD_PAIR = r'\(\s*([^(),]+)\s*,\s*([^(),]+)\s*\)'   # (task, slot)


def input_error(message):
    print("INPUT ERROR:", message)
    return None


def text_lines(filename):
    if not os.path.exists(filename):
        return input_error("file not found: " + filename)
    clean = []
    with open(filename) as file:
        for line in file:
            text = line.strip()
            if text == "":
                continue
            if text.startswith("#"):
                continue
            clean.append(text)
    return clean


def count_at(lines, index):
    if index >= len(lines):
        return input_error("file ends before all items were read")
    numbers = re.findall(r'-?\d+', lines[index])
    if len(numbers) == 0:
        return input_error("expected a number in the input file")
    return int(numbers[0])


def take_pairs(lines, start, count, pattern):
    # Tuples may be wrapped over several lines, so keep reading until we have `count`.
    pairs = []
    while len(pairs) < count:
        if start >= len(lines):
            return input_error("file ends before all items were read"), start
        found = re.findall(pattern, lines[start])
        if len(pairs) + len(found) > count:
            return input_error("more items listed than the count allows"), start
        for pair in found:
            pairs.append(pair)
        start += 1
    chosen = pairs[:count]
    return chosen, start


def slot_number(text):
    # 'T3', '3' or 3 -> 3
    if isinstance(text, int):
        answer = text
    else:
        answer = int(re.sub(r'\D', '', text))
    return answer


def inside_grid(cell, grid):
    answer = 0 <= cell[0] < grid["rows"] and 0 <= cell[1] < grid["cols"]
    return answer


def load_environment(filename):
    lines = text_lines(filename)
    if lines is None:
        return None

    sizes = re.findall(r'-?\d+', lines[0])
    if len(sizes) < 2:
        return input_error("first line must be rows,columns")
    rows = int(sizes[0])
    cols = int(sizes[1])
    if rows <= 0 or cols <= 0:
        return input_error("grid size must be positive")

    wall_count = count_at(lines, 1)
    if wall_count is None:
        return None
    wall_pairs, at = take_pairs(lines, 2, wall_count, NUMBER_PAIR)
    if wall_pairs is None:
        return None
    walls = set()
    for pair in wall_pairs:
        walls.add((int(pair[0]), int(pair[1])))

    dirt_count = count_at(lines, at)
    if dirt_count is None:
        return None
    dirt_pairs, at = take_pairs(lines, at + 1, dirt_count, NUMBER_PAIR)
    if dirt_pairs is None:
        return None
    dirt = []
    for pair in dirt_pairs:
        dirt.append((int(pair[0]), int(pair[1])))

    home_pairs, at = take_pairs(lines, at, 1, NUMBER_PAIR)
    if home_pairs is None:
        return None
    home_pair = home_pairs[0]
    home = (int(home_pair[0]), int(home_pair[1]))

    if len(lines) > at:
        return input_error("extra numbers after the required items")

    grid = {}
    grid["rows"] = rows
    grid["cols"] = cols
    grid["obstacles"] = walls
    grid["dirties"] = dirt
    grid["station"] = home

    for cell in walls:
        if not inside_grid(cell, grid):
            return input_error("a wall is outside the grid")
    for cell in dirt:
        if not inside_grid(cell, grid):
            return input_error("a dirty cell is outside the grid")
    if not inside_grid(home, grid):
        return input_error("charging station is outside the grid")
    seen = set()
    for cell in dirt:
        if cell in seen:
            return input_error("two dirty cells share one cell")
        seen.add(cell)
    if home in walls:
        return input_error("charging station is on a wall")
    if home in seen:
        return input_error("charging station is on a dirty cell")
    for cell in dirt:
        if cell in walls:
            return input_error("a dirty cell is on a wall")
    return grid


def load_schedule(filename):
    lines = text_lines(filename)
    if lines is None:
        return None

    total_slots = count_at(lines, 0)
    if total_slots is None:
        return None
    if total_slots <= 0:
        return input_error("number of time slots must be positive")

    before_count = count_at(lines, 1)
    if before_count is None:
        return None
    before_pairs, at = take_pairs(lines, 2, before_count, WORD_PAIR)
    if before_pairs is None:
        return None
    before = []
    for pair in before_pairs:
        first = pair[0].strip().upper()
        second = pair[1].strip().upper()
        before.append((first, second))

    latest_count = count_at(lines, at)
    if latest_count is None:
        return None
    latest_pairs, at = take_pairs(lines, at + 1, latest_count, WORD_PAIR)
    if latest_pairs is None:
        return None
    latest = {}
    for pair in latest_pairs:
        task = pair[0].strip().upper()
        limit = slot_number(pair[1])
        if task in latest and limit >= latest[task]:
            continue
        latest[task] = limit

    if len(lines) > at:
        return input_error("extra rules after the required items")

    rules = {}
    rules["slots"] = total_slots
    rules["before"] = before
    rules["latest"] = latest
    return rules


def tasks(grid):
    names = {}
    number = 1
    for cell in grid["dirties"]:
        key = "D" + str(number)
        names[key] = cell
        number += 1
    return names


def validate_schedule(rules, grid):
    valid = set()
    for task in tasks(grid):
        valid.add(task)
    for first, second in rules["before"]:
        if first not in valid or second not in valid:
            input_error("a rule names a task that does not exist")
            return False
        if first == second:
            input_error("a task cannot come before itself")
            return False
    for task in rules["latest"]:
        if task not in valid:
            input_error("a deadline names a task that does not exist")
            return False
    return True


env = load_environment("Environment_Sample.txt")
rules = load_schedule("Schedule.txt")
print("Grid:", env["rows"], "x", env["cols"], "| walls:", len(env["obstacles"]))
print("Station:", env["station"], "| dirty cells:", tasks(env))
print("Slots:", rules["slots"], "| before:", rules["before"], "| latest:", rules["latest"])

Grid: 8 x 8 | walls: 6
Station: (0, 0) | dirty cells: {'D1': (0, 7), 'D2': (3, 7), 'D3': (7, 7), 'D4': (7, 0)}
Slots: 4 | before: [('D1', 'D2'), ('D2', 'D3'), ('D1', 'D4')] | latest: {'D2': 3, 'D3': 4}


## 2. Graph search (BFS and A*)

State = one cell `(row, col)`. Actions = UP, DOWN, LEFT, RIGHT (cost 1 each).
A cell is legal if it is inside the grid and not a wall.

- **BFS** uses a queue: it explores cells in layers, so it always finds a shortest path.
- **A*** uses a priority list ordered by `distance travelled + Manhattan distance to goal`.
  Manhattan distance never overestimates a 4-direction walk, so A* also finds a shortest path,
  usually after expanding far fewer cells.

In [24]:
env = load_environment("Environment_Sample.txt")


def get_successors(state, grid):
    row, col = state
    result = []
    # (c, r, move)
    moves = [(-1, 0, "UP"), (1, 0, "DOWN"), (0, -1, "LEFT"), (0, 1, "RIGHT")]
    # (dr -> delta row), (dc -> delta column)z
    for dr, dc, action in moves:
        nxt = (row + dr, col + dc)
        inside = 0 <= nxt[0] < grid["rows"] and 0 <= nxt[1] < grid["cols"]
        if inside:
            if nxt not in grid["obstacles"]:
                result.append((nxt, action, 1))
    return result


def goal_test(state, goal_state):
    answer = state == goal_state
    return answer


def path_cost(path):
    if len(path) == 0:
        cost = None
    else:
        cost = len(path) - 1
    return cost


def heuristic(state, goal_state):
    # manhatton distance 
    row_gap = abs(state[0] - goal_state[0])
    col_gap = abs(state[1] - goal_state[1])
    distance = row_gap + col_gap
    return distance


def make_path(parent, goal):
    path = []
    cell = goal
    while cell is not None:
        path.append(cell)
        cell = parent[cell]
    path.reverse()
    return path


def bfs(grid, start, goal):
    queue = deque()
    queue.append(start)
    parent = {}
    parent[start] = None
    expanded = 0

    while len(queue) > 0:
        cell = queue.popleft()
        expanded += 1
        if goal_test(cell, goal):
            found = make_path(parent, goal)
            cost = path_cost(found)
            result = (found, cost, expanded)
            return result
        successors = get_successors(cell, grid)
        for nxt, action, step in successors:
            if nxt not in parent:
                parent[nxt] = cell
                queue.append(nxt)

    failure = (None, None, expanded)
    return failure


def astar(grid, start, goal):
    first_guess = heuristic(start, goal)
    todo = []
    todo.append((first_guess, start))
    parent = {}
    parent[start] = None
    cost_so_far = {}
    cost_so_far[start] = 0
    done = set()
    expanded = 0

    while len(todo) > 0:
        todo.sort()                      # cheapest guess first (small grid: sorting is fine)
        pair = todo.pop(0)
        cell = pair[1]
        if cell in done:
            continue
        done.add(cell)
        expanded += 1

        if goal_test(cell, goal):
            found = make_path(parent, goal)
            cost = path_cost(found)
            result = (found, cost, expanded)
            return result

        successors = get_successors(cell, grid)
        for nxt, action, step in successors:
            new_cost = cost_so_far[cell] + step
            needs_update = False
            if nxt not in cost_so_far:
                needs_update = True
            else:
                if new_cost < cost_so_far[nxt]:
                    needs_update = True
            if needs_update:
                cost_so_far[nxt] = new_cost
                parent[nxt] = cell
                guess = new_cost + heuristic(nxt, goal)
                todo.append((guess, nxt))

    failure = (None, None, expanded)
    return failure


def search(grid, start, goal, algorithm):
    if algorithm.lower() not in ("bfs", "astar", "a*"):
        print("FAILED: unknown algorithm:", algorithm)
        return None, None, 0
    if not inside_grid(start, grid):
        print("no path: start is outside the grid")
        return None, None, 0
    if not inside_grid(goal, grid):
        print("no path: goal is outside the grid")
        return None, None, 0
    if start in grid["obstacles"]:
        print("no path: start is on a wall")
        return None, None, 0
    if goal in grid["obstacles"]:
        print("no path: goal is on a wall")
        return None, None, 0
    if start == goal:
        only = [start]
        result = (only, 0, 0)
        return result
    if algorithm.lower() in ("astar", "a*"):
        result = astar(grid, start, goal)
    else:
        result = bfs(grid, start, goal)
    return result


path, cost, expanded = astar(env, env["station"], env["dirties"][0])
print("A* example path:", path, "| cost:", cost, "| cells expanded:", expanded)

A* example path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7)] | cost: 7 | cells expanded: 8


## 3. Visuals (no extra libraries)

The helpers below render coloured HTML straight in the notebook.
**C** = charging station, **X** = wall, **D** = dirty, **✓** = cleaned,
**·** = route cell, **S/R** = start/finish. The bar chart compares how many
cells BFS and A* had to expand on the same journeys.

In [25]:
from IPython.display import HTML, display

COLORS = {"wall": "#455a64", "station": "#43a047", "dirty": "#e53935",
          "clean": "#00acc1", "free": "#eceff1", "route": "#90caf9"}
SYMBOLS = {"wall": "X", "station": "C", "dirty": "D", "clean": "✓",
           "free": "", "route": "·"}


def draw_grid(grid, path=None, cleaned=(), title=""):
    if path is None:
        route = []
    else:
        route = path

    parts = []
    parts.append("<b>" + title + "</b>")
    parts.append("<table style='border-collapse:collapse;margin:6px 0'>")

    for row in range(grid["rows"]):
        parts.append("<tr>")
        for col in range(grid["cols"]):
            cell = (row, col)
            if cell == grid["station"]:
                kind = "station"
            elif cell in grid["obstacles"]:
                kind = "wall"
            elif cell in cleaned:
                kind = "clean"
            elif cell in grid["dirties"]:
                kind = "dirty"
            elif cell in route:
                kind = "route"
            else:
                kind = "free"

            mark = SYMBOLS[kind]
            if len(route) > 0:
                if cell == route[0]:
                    mark = "S"
                if cell == route[-1]:
                    mark = "R"

            style = ("width:26px;height:26px;text-align:center;color:white;"
                     "font-weight:bold;border:1px solid white;background:")
            style = style + COLORS[kind]
            parts.append("<td style='" + style + "'>" + mark + "</td>")
        parts.append("</tr>")
    parts.append("</table>")

    page = ""
    for part in parts:
        page += part
    display(HTML(page))


def draw_comparison(names, bfs_counts, astar_counts, title):
    biggest = 0
    for n in bfs_counts:
        if n > biggest:
            biggest = n
    for n in astar_counts:
        if n > biggest:
            biggest = n
    if biggest == 0:
        biggest = 1

    parts = []
    parts.append("<b>" + title + "</b>")
    parts.append("<table style='font-size:12px;border-collapse:collapse'>")
    parts.append("<tr><th align=left>Journey</th><th>BFS</th><th>A*</th></tr>")

    index = 0
    while index < len(names):
        name = names[index]
        b = bfs_counts[index]
        a = astar_counts[index]
        bw = round(140 * b / biggest)
        aw = round(140 * a / biggest)
        line = "<tr><td>" + str(name) + "</td>"
        line += "<td><div style='background:#42a5f5;color:white;width:" + str(bw) + "px'>" + str(b) + "</div></td>"
        line += "<td><div style='background:#66bb6a;color:white;width:" + str(aw) + "px'>" + str(a) + "</div></td></tr>"
        parts.append(line)
        index += 1
    parts.append("</table>")

    page = ""
    for part in parts:
        page += part
    display(HTML(page))


draw_grid(env, title="Our environment before cleaning")

C,,,,,,,D
,,,X,,,,
,,,X,,,,
,,,X,,,,D
,,,X,,,,
,,,X,,,,
,,,X,,,,
D,,,,,,,D


## 4. CSP scheduling

Variables = the tasks `D1..Dn` (one per dirty cell, even if unmentioned in the rules).
Domains = available slots `{1..n}`, trimmed by any "no later than" rule.
Constraints = **before** (precedence), **latest**, and **one task per slot**.

Backtracking picks the variable with the fewest remaining values (MRV; ties broken by the
most links to other tasks), tries values that rule out the fewest choices (LCV), and removes
impossible values after each choice (forward checking). Fresh domain copies make undo automatic.

In [27]:
rules = load_schedule("Schedule_Sample.txt")


def all_variables(grid):
    variables = []
    number = 1
    while number <= len(grid["dirties"]):
        variables.append("D" + str(number))
        number += 1
    return variables


def starting_domains(variables, rules):
    domains = {}
    for var in variables:
        if var in rules["latest"]:
            last = rules["latest"][var]
        else:
            last = rules["slots"]
        if last > rules["slots"]:
            last = rules["slots"]
        allowed = set()
        for number in range(1, last + 1):
            allowed.add(number)
        domains[var] = allowed
    return domains


def linked_to(var, rules):
    friends = set()
    for first, second in rules["before"]:
        if first == var:
            friends.add(second)
        if second == var:
            friends.add(first)
    return friends


def select_unassigned_variable(assignment, domains, rules):
    free = []
    free_set = set()
    for var in domains:
        if var not in assignment:
            free.append(var)
            free_set.add(var)

    best = free[0]
    for var in free:
        choose = False
        if len(domains[var]) < len(domains[best]):
            choose = True
        else:
            if len(domains[var]) == len(domains[best]):
                degree_var = len(linked_to(var, rules) & free_set)
                degree_best = len(linked_to(best, rules) & free_set)
                if degree_var > degree_best:
                    choose = True
                else:
                    if degree_var == degree_best:
                        if var < best:
                            choose = True
        if choose:
            best = var
    return best


def order_domain_values(variable, assignment, domains, rules):
    # Try the slot that spoils the fewest options for other tasks first.
    def damage(value):
        count = 0
        for other in domains:
            if other == variable:
                continue
            if other in assignment:
                continue
            for slot in domains[other]:
                if slot == value:
                    count += 1
                if (variable, other) in rules["before"] and value >= slot:
                    count += 1
                if (other, variable) in rules["before"] and slot >= value:
                    count += 1
        return count

    ranked = []
    for value in domains[variable]:
        ranked.append((damage(value), value))
    ranked.sort()

    ordered = []
    for pair in ranked:
        ordered.append(pair[1])
    return ordered


def is_consistent(variable, value, assignment, rules):
    if value in assignment.values():          # slot already taken
        ok = False
    else:
        if value > rules["latest"].get(variable, rules["slots"]):
            ok = False
        else:
            ok = True
            for first, second in rules["before"]:
                if first == variable:
                    if second in assignment:
                        if value >= assignment[second]:
                            ok = False
                if second == variable:
                    if first in assignment:
                        if assignment[first] >= value:
                            ok = False
    return ok


def forward_check(variable, value, assignment, domains, rules):
    new = {}                                  # copy = free undo on backtrack
    for var in domains:
        slots = set()
        for slot in domains[var]:
            slots.add(slot)
        new[var] = slots

    for other in new:
        if other == variable:
            continue
        if other in assignment:
            continue
        new[other].discard(value)              # slot exclusivity

        for first, second in rules["before"]:
            if first == variable:
                if second == other:
                    kept = set()
                    for slot in new[other]:
                        if slot > value:
                            kept.add(slot)
                    new[other] = kept
            if second == variable:
                if first == other:
                    kept = set()
                    for slot in new[other]:
                        if slot < value:
                            kept.add(slot)
                    new[other] = kept

        if len(new[other]) == 0:
            failure = None
            return failure

    return new


def backtracking_search(variables, domains, rules):
    assignment = {}

    def solve(domains):
        if len(assignment) == len(variables):
            answer = {}
            for key in assignment:
                answer[key] = assignment[key]
            return answer

        var = select_unassigned_variable(assignment, domains, rules)
        choices = order_domain_values(var, assignment, domains, rules)
        for value in choices:
            if not is_consistent(var, value, assignment, rules):
                continue
            smaller = forward_check(var, value, assignment, domains, rules)
            if smaller is None:
                continue
            assignment[var] = value
            answer = solve(smaller)
            if answer is not None:
                return answer
            del assignment[var]

        failure = None
        return failure

    result = solve(domains)
    return result


def solve_schedule(grid, rules):
    # Accept both 'T3' and 3 in hand-written rule dictionaries too.
    latest = {}
    for key in rules["latest"]:
        latest[key] = slot_number(rules["latest"][key])

    clean_rules = {}
    clean_rules["slots"] = rules["slots"]
    clean_rules["before"] = rules["before"]
    clean_rules["latest"] = latest

    variables = all_variables(grid)
    if clean_rules["slots"] < len(variables):
        result = None
    else:
        domains = starting_domains(variables, clean_rules)
        result = backtracking_search(variables, domains, clean_rules)
    return result


def show_schedule(assignment, grid):
    if assignment is None:
        print("NO FEASIBLE SCHEDULE")
        return None

    print(" slot | task | dirty cell")
    by_slot = []
    for task in assignment:
        by_slot.append((assignment[task], task))
    by_slot.sort()
    for pair in by_slot:
        slot = pair[0]
        task = pair[1]
        print("  T" + str(slot) + "  |  " + task + "   | " + str(tasks(grid)[task]))
    print("all tasks assigned once, before/latest rules checked, slots unique")
    return None


assignment = solve_schedule(env, rules)
show_schedule(assignment, env)

 slot | task | dirty cell
  T1  |  D2   | (3, 7)
  T2  |  D1   | (0, 7)
  T3  |  D3   | (7, 7)
  T4  |  D4   | (7, 0)
all tasks assigned once, before/latest rules checked, slots unique


## 5. Integration and test runs

`execute_schedule` turns the CSP assignment into a cleaning order, calls the chosen search
for every leg (including the trip home), joins paths without duplicating shared end cells,
and reports failure the moment a leg is impossible.

In [28]:
def execute_schedule(grid, charging_station, schedule, algorithm):
    by_slot = []
    for task in schedule:
        by_slot.append((schedule[task], task))
    by_slot.sort()

    journey = []
    for pair in by_slot:
        task = pair[1]
        journey.append(tasks(grid)[task])
    journey.append(charging_station)

    here = charging_station
    complete_path = []
    complete_path.append(here)
    total_cost = 0

    for stop in journey:
        leg, cost, expanded = search(grid, here, stop, algorithm)
        if leg is None:
            result = (complete_path, total_cost, False)
            return result
        position = 1                          # skip leg[0]: already in complete_path
        while position < len(leg):
            complete_path.append(leg[position])
            position += 1
        total_cost += cost
        here = stop

    cleaned_all = True
    for cell in grid["dirties"]:
        if cell not in journey:
            cleaned_all = False
    if here == charging_station and cleaned_all:
        success = True
    else:
        success = False

    result = (complete_path, total_cost, success)
    return result


## 6. Test runs

Four runs: our own files, the provided sample, an impossible schedule, and an unreachable
dirty cell. Each run prints the schedule, per-leg BFS/A* costs and expanded cells, the full
route and status, then draws the grid and the comparison chart.

In [37]:
def test_case(env_file, schedule_file, algorithm="astar"):
    grid = load_environment(env_file)
    if grid is None:
        return
    rule_book = load_schedule(schedule_file)
    if rule_book is None:
        return
    if not validate_schedule(rule_book, grid):
        return
    print("\n" + "=" * 70)
    print("TEST:", env_file, "+", schedule_file, "(navigation:", algorithm + ")")
    print("=" * 70)

    plan = solve_schedule(grid, rule_book)
    show_schedule(plan, grid)
    if plan is None:
        draw_grid(grid, title=env_file + ": no feasible schedule")
        return None

    by_slot = []
    for task in plan:
        by_slot.append((plan[task], task))
    by_slot.sort()
    order = []
    for pair in by_slot:
        order.append(tasks(grid)[pair[1]])

    order_text = ""
    for cell in order:
        if order_text != "":
            order_text += " -> "
        order_text += str(cell)
    print("Cleaning order:", order_text)

    journey = []
    for stop in order:
        journey.append(stop)
    journey.append(grid["station"])

    names = []
    bfs_nodes = []
    astar_nodes = []
    here = grid["station"]
    for stop in journey:
        found_b, b_cost, b_count = bfs(grid, here, stop)
        found_a, a_cost, a_count = astar(grid, here, stop)
        label = str(here[0]) + "," + str(here[1]) + " to " + str(stop[0]) + "," + str(stop[1])
        names.append(label)
        bfs_nodes.append(b_count)
        astar_nodes.append(a_count)

        if b_cost is None:
            b_text = "no path"
        else:
            b_text = str(b_cost) + " moves"
        if a_cost is None:
            a_text = "no path"
        else:
            a_text = str(a_cost) + " moves"
        print("  " + label.rjust(18) + ": BFS " + b_text + " / " + str(b_count) +
              " cells | A* " + a_text + " / " + str(a_count) + " cells")
        here = stop

    draw_comparison(names, bfs_nodes, astar_nodes, "BFS vs A* - cells expanded")

    full_path, total, success = execute_schedule(grid, grid["station"], plan, algorithm)
    print("Route:", full_path)
    print("Total movements:", path_cost(full_path), "| Total navigation cost:", total)
    if success:
        print("Status: SUCCESS - all cells cleaned and robot back at station.")
    else:
        print("FAILED: no path - a required leg could not be completed.")

    cleaned = set()
    if success:
        for stop in order:
            cleaned.add(stop)
    if success:
        title = "Journey - " + str(total) + " moves"
    else:
        title = "Journey stops at the failure - " + str(total) + " moves"
    draw_grid(grid, full_path, cleaned, title)
    return None


test_case("Environment.txt", "Schedule.txt")                     # our own files
test_case("Environment_Sample.txt", "Schedule_Sample.txt")       # provided sample
test_case("Environment.txt", "Schedule_Error.txt")               # infeasible CSP
test_case("Environment_Error.txt", "Schedule.txt")               # unreachable dirt



TEST: Environment.txt + Schedule.txt (navigation: astar)
 slot | task | dirty cell
  T1  |  D1   | (1, 7)
  T2  |  D2   | (4, 7)
  T3  |  D3   | (7, 6)
  T4  |  D4   | (6, 0)
all tasks assigned once, before/latest rules checked, slots unique
Cleaning order: (1, 7) -> (4, 7) -> (7, 6) -> (6, 0)
          0,0 to 1,7: BFS 8 moves / 38 cells | A* 8 moves / 15 cells
          1,7 to 4,7: BFS 3 moves / 10 cells | A* 3 moves / 4 cells
          4,7 to 7,6: BFS 4 moves / 18 cells | A* 4 moves / 7 cells
          7,6 to 6,0: BFS 7 moves / 38 cells | A* 7 moves / 8 cells
          6,0 to 0,0: BFS 6 moves / 24 cells | A* 6 moves / 7 cells


Journey,BFS,A*
"0,0 to 1,7",38,15
"1,7 to 4,7",10,4
"4,7 to 7,6",18,7
"7,6 to 6,0",38,8
"6,0 to 0,0",24,7


Route: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 7), (2, 7), (3, 7), (4, 7), (4, 6), (5, 6), (6, 6), (7, 6), (6, 6), (6, 5), (6, 4), (6, 3), (6, 2), (6, 1), (6, 0), (5, 0), (4, 0), (3, 0), (2, 0), (1, 0), (0, 0)]
Total movements: 28 | Total navigation cost: 28
Status: SUCCESS - all cells cleaned and robot back at station.


R,·,·,·,·,·,·,·
·,,X,,,,,✓
·,,X,,,,,·
·,,X,,,,,·
·,,X,,,X,·,✓
·,,X,,,X,·,
✓,·,·,·,·,·,·,
,,,,,,✓,



TEST: Environment_Sample.txt + Schedule_Sample.txt (navigation: astar)
 slot | task | dirty cell
  T1  |  D2   | (3, 7)
  T2  |  D1   | (0, 7)
  T3  |  D3   | (7, 7)
  T4  |  D4   | (7, 0)
all tasks assigned once, before/latest rules checked, slots unique
Cleaning order: (3, 7) -> (0, 7) -> (7, 7) -> (7, 0)
          0,0 to 3,7: BFS 10 moves / 48 cells | A* 10 moves / 29 cells
          3,7 to 0,7: BFS 3 moves / 10 cells | A* 3 moves / 4 cells
          0,7 to 7,7: BFS 7 moves / 27 cells | A* 7 moves / 8 cells
          7,7 to 7,0: BFS 7 moves / 33 cells | A* 7 moves / 8 cells
          7,0 to 0,0: BFS 7 moves / 26 cells | A* 7 moves / 8 cells


Journey,BFS,A*
"0,0 to 3,7",48,29
"3,7 to 0,7",10,4
"0,7 to 7,7",27,8
"7,7 to 7,0",33,8
"7,0 to 0,0",26,8


... (4 outputs left out)


TEST: Environment_Error.txt + Schedule.txt (navigation: astar)
 slot | task | dirty cell
  T1  |  D1   | (0, 7)
  T2  |  D2   | (3, 7)
  T3  |  D3   | (7, 7)
  T4  |  D4   | (6, 1)
all tasks assigned once, before/latest rules checked, slots unique
Cleaning order: (0, 7) -> (3, 7) -> (7, 7) -> (6, 1)
          0,0 to 0,7: BFS 7 moves / 30 cells | A* 7 moves / 8 cells
          0,7 to 3,7: BFS 3 moves / 7 cells | A* 3 moves / 4 cells
          3,7 to 7,7: BFS 4 moves / 20 cells | A* 4 moves / 5 cells
          7,7 to 6,1: BFS no path / 55 cells | A* no path / 55 cells
          6,1 to 0,0: BFS no path / 1 cells | A* no path / 1 cells


Journey,BFS,A*
"0,0 to 0,7",30,8
"0,7 to 3,7",7,4
"3,7 to 7,7",20,5
"7,7 to 6,1",55,55
"6,1 to 0,0",1,1


Route: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 7), (2, 7), (3, 7), (4, 7), (5, 7), (6, 7), (7, 7)]
Total movements: 14 | Total navigation cost: 14
FAILED: no path - a required leg could not be completed.


S,·,·,·,·,·,·,D
,,,,,,,·
,,,,,,,·
,,,,,,,D
,,,,,,,·
X,X,X,,,,,·
X,D,X,,,,,·
X,X,X,,,,,R
